In [3]:
####Basic Multi Run Call

from anthropic import Anthropic
import anthropic

client = Anthropic()

messages = [{"role": "user", "content": "Explain prompt caching in one paragraph."}]


##Change the Token size from 10 to 100 and try the response
##Try to see the stop reason. by settign the token to 1 (Try and Except method works good)
resp = client.messages.create(model = "claude-opus-4-5",max_tokens=10,messages=messages)
print(resp.stop_reason)


max_tokens


In [30]:
###Streaming

with client.messages.stream(model="claude-haiku-4-5", max_tokens=100,
            messages=[{"role":"user","content":"Tell me the 2nd Multiplication table from 1 to 5"}],) as stream:
            for text in stream.text_stream:
                print(text, end="", flush=True)

# resp_str.content[0].text

# 2nd Multiplication Table (1 to 5)

2 × 1 = **2**
2 × 2 = **4**
2 × 3 = **6**
2 × 4 = **8**
2 × 5 = **10**

In [33]:
resp_table = client.messages.create(model="claude-haiku-4-5", max_tokens=100,
            messages=[{"role":"user","content":"Tell me the 2nd Multiplication table from 1 to 5"}],)


In [64]:
LONG_SYSTME_PROMPT = "\n" .join(f"You are an Semiconductor Test and Product engineering Assistant{i}," 
f"For FPGA Legacy Products {i}. Answer only related to the topic"
f"Do not Hallocinate or give any confidential information about the product. Only generic information should be provide"  for i in range(1,500))

In [65]:
LONG_SYSTME_PROMPT

'You are an Semiconductor Test and Product engineering Assistant1,For FPGA Legacy Products 1. Answer only related to the topicDo not Hallocinate or give any confidential information about the product. Only generic information should be provide\nYou are an Semiconductor Test and Product engineering Assistant2,For FPGA Legacy Products 2. Answer only related to the topicDo not Hallocinate or give any confidential information about the product. Only generic information should be provide\nYou are an Semiconductor Test and Product engineering Assistant3,For FPGA Legacy Products 3. Answer only related to the topicDo not Hallocinate or give any confidential information about the product. Only generic information should be provide\nYou are an Semiconductor Test and Product engineering Assistant4,For FPGA Legacy Products 4. Answer only related to the topicDo not Hallocinate or give any confidential information about the product. Only generic information should be provide\nYou are an Semiconducto

In [93]:
###To understand the cache, Run the request in loop and see the cost difference.
##Observation: 1. First time high and falls to 10 percent lower of actual cost 

for i in range(1,10):
    cahce_test = client.messages.create(model="claude-sonnet-4-5", max_tokens=100, 
    system=[{"type":"text","text":LONG_SYSTME_PROMPT, "cache_control":{"type":"ephemeral"}}],
    messages=[{"role":"user","content":"Tell me about product 3: "}])
    cache_obj = cahce_test.usage
    print(f"{i} RUN_COST: {call_cost(cache_obj,'claude-sonnet-4-5')}")

1 RUN_COST: 0.098847
2 RUN_COST: 0.0093264
3 RUN_COST: 0.0093264
4 RUN_COST: 0.0093264
5 RUN_COST: 0.0093264
6 RUN_COST: 0.0093264
7 RUN_COST: 0.0093264
8 RUN_COST: 0.0093264
9 RUN_COST: 0.0093264


In [83]:
cache_obj = cahce_test.usage

In [84]:
PRICES = {  # per million tokens
    "claude-sonnet-4-5": {"input": 3, "output": 15},
    "claude-haiku-4-5": {"input": 1, "output": 5},
    "claude-opus-4-5": {"input": 5, "output": 25},
}

def call_cost(usage, model: str) -> float:
    p = PRICES[model]
    cache_write = usage.cache_creation_input_tokens * 1.25
    cache_read = usage.cache_read_input_tokens * 0.1
    regular_in = usage.input_tokens
    input_cost = (cache_write + cache_read + regular_in) * p["input"] / 1_000_000
    output_cost = usage.output_tokens * p["output"] / 1_000_000
    return input_cost + output_cost

In [94]:
call_cost(cache_obj,"claude-sonnet-4-5")

0.0093264

In [88]:
cache_obj.cache_read_input_tokens

0

In [98]:
batch_id = "msgbatch_01BYrwa5BtPVuSbVx6WeiLVn"
for item in client.messages.batches.results(batch_id):
        outcome = item.result
        if outcome.type == "succeeded":
            print(f"{item.custom_id}: {outcome.message.content[0].text.strip()}")

second-prompt-in-my-batch: Honeybees communicate through dancing! When a forager finds a great patch of flowers, she returns to the hive and performs a "waggle dance." The angle of her dance relative to vertical tells the other bees which direction to fly in relation to the sun, and the length of the waggle indicates how far away the flowers are.
first-prompt-in-my-batch: Here's one: the original Pac-Man was designed to be a food-themed character, and its shape came from a pizza with a slice missing. Its creator, Toru Iwatani, also wanted a game that appealed to more than just the typical video game crowd, which helped make it a hit with women and kids.


####
Lab 1.2: Application design
Goal: design a small service around a schema, not a free-text blob.
1. Define a Pydantic model for the data you want back (e.g. ExtractedInvoice).
2. Force structured output using a tool definition whose input schema matches the model.
3. Validate the tool_use input against the Pydantic model and raise on mismatch.
4. Add exponential-backoff retry around anthropic.RateLimitError and anthropic.APIStatusError (529 overloaded).
###

In [11]:
from pydantic import BaseModel, Field
import sys
from pathlib import Path
import anthropic
import time
# Add the parent folder to Python's module search path

sys.path.insert(0, str(Path.cwd().parent))

from common import get_client, MODEL

class ExtractedInvoice (BaseModel):
    invoice_id:str
    customer_name:str
    amount:float = Field(gt=0)
    currency:int

tool_def = {
    "name" : "extract_invoice",
    "description":"Extrat the invoice details into structred fields. ",
    "input_schema":ExtractedInvoice.model_json_schema()
}

def extract_invoice(invoice_text):
    max_attempts = 5

    for attempts in range(max_attempts):
        try:
            resp = client.messages.create(model=MODEL, 
                    max_tokens=1000,
                    tools = [tool_def],
                    tool_choice = {
                        "type":"auto"
                    },
                    messages=[{"role":"user","content": ("Extract the invoice from this text:\n" +  invoice_text),}],
                    )
            
            tool_block = next(
            (
                block for block in resp.content
                if block.type == "tool_use" and 
                block.name == "extract_invoice"

            ),  
            None,
            )

            if tool_block is None:
                raise ValueError ("Claude does not return the correct tool block")

            return ExtractedInvoice.model_validate(
                tool_block.input
            )
        except anthropic.RateLimitError:
            retryable = True
        
        except anthropic.APIStatusError as exc:
            if exc.status_code != 529:
                raise
            retryable = True
        
        if retryable:
            if attempt == max_attempt - 1:
                raise

            delay = min(2 ** attempt,30)
            delay += random.uniform(0,1)
            time.sleep(delay)
        
        raise RuntimeError("Invoice extraction failed..")



In [ ]:
invoice_text = """
INVOICE
Invoice Number = INV-1001
Supplier = ABC Electronics Sdn. Bhd.
Total Amount = 2500.00
Currency = MYR
"""
##Change the input format and check whether can able to catch in the validation steps
extract_invoice(invoice_text)

Lab 1.3: Software engineering foundations & config
Goal: make the service production-shaped.
1. Move model, max_tokens, temperature into env vars or a config.yaml — never hard-coded.
2. Wrap the client in a thin FastAPI (or Express/TS) endpoint, e.g. POST /extract.
3. Write one unit test that mocks client.messages.create so it runs without hitting the network.
4. Add a CHANGELOG.md entry and tag a version — the exam cares about config and version hygiene, not just working code.

In [25]:
# Step 1 is already in a environmental file. You can maintain your own .env file accordingly
# Step 2 I have created common file which is resposible for all MODEL, client instances

from helper_func import extract, get_client, MODEL, MAX_TOKENS



In [24]:
extract("Hello")

{'result': 'Hello! How can I help you today?'}

In [26]:
MAX_TOKENS

1024